# ESCM 2.0 — moving-edge baseline

**Question.** Does the current ESCM retain a newly stimulated edge after camera input ends, and suppress previously stimulated locations as the edge moves?

This reproduces the *type of stimulus* described in the paper's 25-neuron tuning experiment (Fig. 3): a five-pixel-wide edge advances across a 5 × 5 network. The paper does not specify all individual event times, so the schedule below is an explicit test design, **not** a claim of reproducing the published numerical figure.

We start with the **existing Brian2 mechanism**, without redesigning it. The original experiment sought self-sustained firing at the final edge and suppression of the preceding positions. The resulting spike raster and membrane/persistence traces are our qualitative baseline for ESCM 2.0.

The default configuration uses the **legacy script** parameters (threshold 0.1, input weight 0.5, exact integration), matching the current refactored notebook. Change `parameter_set` to `"paper"` to examine the published parameter values (threshold 1.0, input weight 1.1); this is a different experiment.

## 1. Moving edge and model parameters

A five-neuron vertical edge appears in successive columns, with **one input spike per pixel**. Columns are stimulated 4 ms apart. We then provide 60 ms with no input to observe whether the final edge persists.

The 5 × 5 grid is deliberately small: for `k=8`, every other pixel is a neighbor. We exclude autapses from recurrent connections, as in the current Brian2 implementation. This notebook does not attempt to reconstruct the full recording pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *

parameter_set = "legacy"   # "legacy" or "paper"
parameters = {
    "legacy": {"vt": 0.1, "beta": 0.5},
    "paper": {"vt": 1.0, "beta": 1.1},
}
vt = parameters[parameter_set]["vt"]
beta = parameters[parameter_set]["beta"]
Wi, Wk, k = 6.0, -3.0, 8
tau, vr = 0.2*ms, 0.0
width = height = 5
N = width*height
dt = 0.5*ms
step_ms = 4.0
start_ms = 1.0
silence_ms = 60.0

# Row-major neuron indices: index = y*width + x.
input_ids = np.array([row*width + column
                      for column in range(width)
                      for row in range(height)], dtype=np.int64)
input_times_ms = np.repeat(start_ms + step_ms*np.arange(width), height)
last_input_ms = float(input_times_ms.max())
run_ms = last_input_ms + silence_ms
print(f"Parameters: {parameter_set}, threshold={vt}, beta={beta}")
print(f"Stimulus columns at {start_ms + step_ms*np.arange(width)} ms")
print(f"Last input at {last_input_ms:g} ms; simulation ends at {run_ms:g} ms")

## 2. Simulate the current ESCM

The membrane equation and three decaying input states are

\[
\tau\dot v_i=-v_i+\mathrm{ExtIn}_i+P_i(\mathrm{Exc}_i+\mathrm{Inh}_i),\quad
\tau\dot z_i=-z_i,\ z\in\{\mathrm{ExtIn},\mathrm{Exc},\mathrm{Inh}\}.
\]

A camera spike **sets** \(\mathrm{ExtIn}=\beta\). A recurrent spike sets excitation at its source and increments neighbor spike counters before applying inhibition. Custom events control \(P\). We retain the corrected neuron-local counter reset and explicitly ordered recurrent pathways.

This preserves the **current model's computation**. It does not establish that the model follows every idealized equation in the paper.

In [ ]:
start_scope()
defaultclock.dt = dt
camera = SpikeGeneratorGroup(N, input_ids, input_times_ms*ms)

equations = """
dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau : 1
dExtIn/dt = -ExtIn/tau : 1
dExc/dt = -Exc/tau : 1
dInh/dt = -Inh/tau : 1
incoming_spikes : integer
P : integer
"""
neurons = NeuronGroup(
    N, equations, threshold="v > vt",
    reset="v = vr; incoming_spikes = 0", refractory=0*ms,
    events={"P_ON": "v > vt",
            "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt)"},
    method="exact",
)
neurons.set_event_schedule("P_ON", when="after_thresholds")
neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
neurons.set_event_schedule("P_OFF", when="groups")
neurons.run_on_event("P_OFF", "P = 0", when="groups")

feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta")
feed.connect(j="i")
recurrent = Synapses(
    neurons, neurons,
    on_pre={"pre": "incoming_spikes_post += 1; Exc_pre = Wi",
            "pre_2": "Inh_post = P_post * Wk * incoming_spikes_post"},
)
recurrent.pre.order = 0
recurrent.pre_2.order = 1
pre = np.repeat(np.arange(N), N)
post = np.tile(np.arange(N), N)
neighbors = ((pre != post)
             & (np.abs(pre % width - post % width) <= k)
             & (np.abs(pre // width - post // width) <= k))
recurrent.connect(i=pre[neighbors], j=post[neighbors])

input_spikes = SpikeMonitor(camera)
output_spikes = SpikeMonitor(neurons)
states = StateMonitor(neurons, ["v", "P", "ExtIn", "Exc", "Inh", "incoming_spikes"],
                      record=True, dt=dt)

run(run_ms*ms)
print(f"Input spikes: {input_spikes.num_spikes}")
print(f"Output spikes: {output_spikes.num_spikes}")
print(f"Output after last input: {np.count_nonzero(np.asarray(output_spikes.t/ms) > last_input_ms)}")

## 3. Spatial raster: does the edge travel and persist?

Each neuron index maps to \((x=i\bmod 5, y=\lfloor i/5\rfloor)\). The top panel shows camera events; the bottom panel shows output spikes. Vertical reference lines indicate presentation of successive columns. Activity past the final reference line indicates **continued output after camera stimulation**, but does not by itself establish correct lateral suppression.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)
for ax, mon, title in zip(axes, (input_spikes, output_spikes),
                          ("Camera events", "ESCM output spikes")):
    t_ms = np.asarray(mon.t/ms)
    ids = np.asarray(mon.i)
    ax.scatter(t_ms, ids, s=9, alpha=0.7)
    for column, onset in enumerate(start_ms + step_ms*np.arange(width)):
        ax.axvline(onset, color="0.65", linestyle=":", linewidth=1)
    ax.axvline(last_input_ms, color="black", linestyle="--", linewidth=1,
               label="Last input" if ax is axes[1] else None)
    ax.set(ylabel="Neuron index (row × 5 + column)", title=title, ylim=(-1, N))
axes[1].set(xlabel="Model time (ms)", xlim=(0, run_ms))
axes[1].legend(loc="upper right")
plt.show()

## 4. Identify whether the final edge dominates

We count output spikes *after input ends*, split by column. In the intended moving-edge behavior, the last column should remain active more strongly than earlier columns. This is a **functional indicator**, not a strict theorem: whether it holds depends on stimulation intervals, inhibition, and parameter values.

In [ ]:
out_t = np.asarray(output_spikes.t/ms)
out_i = np.asarray(output_spikes.i, dtype=int)
after = out_t > last_input_ms + float(dt/ms)
counts = np.bincount(out_i[after] % width, minlength=width)
print("Output spikes per column after last input:")
for column, count in enumerate(counts):
    print(f"  Column {column}: {count:>6,d}")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(width), counts)
ax.set(xlabel="Edge position (column)", ylabel="Output spikes after final input",
       title="Which columns remain active without input?", xticks=np.arange(width))
plt.show()
print("Last column is most active:", bool(counts[-1] >= np.max(counts[:-1])))

## 5. Internal states: explain the behavior

Inspect a neuron in the first, middle, and last columns (all from the central row). The plots reveal whether persistence \(P\) stays enabled, whether self-excitation sustains the membrane potential, and whether neighboring activity drives inhibition. Because Brian2's state monitor samples at its defined schedule, discontinuous synaptic assignments may be visible at the **next** sampled timepoint.

In [ ]:
t_ms = np.asarray(states.t/ms)
representatives = [2*width + column for column in (0, 2, 4)]
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
for i in representatives:
    label = f"column {i % width}"
    axes[0].plot(t_ms, np.asarray(states.v[i]), label=label)
    axes[1].step(t_ms, np.asarray(states.P[i]), where="post", label=label)
    axes[2].plot(t_ms, np.asarray(states.Exc[i]), label=label)
    axes[3].plot(t_ms, np.asarray(states.Inh[i]), label=label)
axes[0].axhline(vt, color="black", linestyle="--", linewidth=1, label="threshold")
for ax, name in zip(axes, ("Membrane potential v", "Persistence gate P",
                            "Excitatory trace Exc", "Inhibitory trace Inh")):
    ax.set_ylabel(name)
    ax.grid(alpha=0.2)
axes[0].legend(ncol=4)
axes[-1].set_xlabel("Model time (ms)")
plt.show()

## 6. Interpretation and next change

This is the **baseline**, not a new implementation. Record the output for both `parameter_set="legacy"` and `"paper"`; do not combine them into one equivalence claim.

The desirable behavior is: (i) camera events trigger corresponding spikes, (ii) activity continues when input stops, and (iii) the most recently stimulated edge preferentially survives. If one condition fails, use the membrane, \(P\), excitation and inhibition traces to identify why. Only then test a simplified ESCM 2.0 mechanism against this baseline.